# NB_03 · Développement des modèles (version corrigée)
### Prédiction d'annulation de réservations d'hôtel

**Objectif :** entraîner plusieurs classificateurs dans des conditions comparables, sur les
variables **originales** et sur les composantes **PCA**, et mesurer leur temps d'entraînement.

**Corrections apportées par rapport à la version précédente :**
- Chargement des matrices **`.npz`** produites par `NB_02_Pretraitement` (Original **et** PCA)
- **8 modèles** entraînés au lieu de 3 : Logistic Regression, Decision Tree (Entropy),
  Decision Tree (Gini), Random Forest — chacun sur les deux versions de données
- Arbres volontairement peu profonds (`max_depth=3`) pour rester interprétables
- `class_weight="balanced"` pour compenser le déséquilibre 63 %/37 % de la cible
- Chronométrage de chaque entraînement, sauvegardé dans un **manifeste** (`model_manifest.csv`)


In [1]:
from pathlib import Path
import time
import re
import joblib
import pandas as pd
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train_original = sparse.load_npz(PROCESSED_DIR / "X_train.npz")
X_train_pca = sparse.load_npz(PROCESSED_DIR / "X_train_pca.npz")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["is_canceled"]

print("Train original :", X_train_original.shape)
print("Train PCA      :", X_train_pca.shape)
print("Cible          :", y_train.shape)


Train original : (69782, 556)
Train PCA      : (69782, 9)
Cible          : (69782,)


## 1. Définition des modèles

Les hyperparamètres principaux sont maintenus identiques entre les variantes Original et PCA,
afin que la comparaison soit équitable. Les arbres sont volontairement limités à 3 niveaux de
profondeur pour rester interprétables. `class_weight="balanced"` évite qu'un arbre peu profond
ignore complètement la classe minoritaire (les annulations).

In [2]:
model_specs = [
    ("Logistic Regression", "Original", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"
    )),
    ("Decision Tree - Entropy", "Original", DecisionTreeClassifier(
        criterion="entropy", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Decision Tree - Gini", "Original", DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Random Forest", "Original", RandomForestClassifier(
        n_estimators=20, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced_subsample", min_samples_leaf=3,
        max_depth=18, max_features="sqrt"
    )),
    ("Logistic Regression - PCA", "PCA", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"
    )),
    ("Decision Tree - Entropy - PCA", "PCA", DecisionTreeClassifier(
        criterion="entropy", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Decision Tree - Gini - PCA", "PCA", DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Random Forest - PCA", "PCA", RandomForestClassifier(
        n_estimators=20, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced_subsample", min_samples_leaf=3,
        max_depth=18, max_features="sqrt"
    ))
]

print("Nombre de modèles à entraîner :", len(model_specs))


Nombre de modèles à entraîner : 8


**Pourquoi ces choix ?**
- **Logistic Regression** : modèle linéaire simple, sert de référence (*baseline*).
- **Decision Tree (Entropy vs Gini)** : deux critères de division différents, comparés pour voir
  si le choix du critère change la performance sur ce dataset.
- **Random Forest** : agrégation de plusieurs arbres, réduit la variance d'un arbre seul.
- **`class_weight="balanced"`** : recalcule automatiquement le poids de chaque classe en fonction
  de sa fréquence, pour que le modèle ne favorise pas simplement la classe majoritaire (non annulée).

## 2. Entraînement, chronométrage et sauvegarde

In [3]:
training_rows = []

for name, dataset_version, model in model_specs:
    X_fit = X_train_pca if dataset_version == "PCA" else X_train_original
    print(f"Entraînement : {name} ({dataset_version})")

    start = time.perf_counter()
    model.fit(X_fit, y_train)
    elapsed = time.perf_counter() - start

    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    path = MODEL_DIR / f"{slug}.joblib"
    joblib.dump(model, path)

    training_rows.append({
        "Modèle": name,
        "Famille": name.replace(" - PCA", ""),
        "Version_données": dataset_version,
        "Fichier": path.name,
        "Temps_entraînement_secondes": elapsed,
        "Nombre_variables": X_fit.shape[1]
    })

training_manifest = pd.DataFrame(training_rows)
assert training_manifest["Temps_entraînement_secondes"].notna().all()
training_manifest.to_csv(MODEL_DIR / "model_manifest.csv", index=False)
training_manifest.round(4)


Entraînement : Logistic Regression (Original)


Entraînement : Decision Tree - Entropy (Original)
Entraînement : Decision Tree - Gini (Original)


Entraînement : Random Forest (Original)


Entraînement : Logistic Regression - PCA (PCA)
Entraînement : Decision Tree - Entropy - PCA (PCA)


Entraînement : Decision Tree - Gini - PCA (PCA)


Entraînement : Random Forest - PCA (PCA)


,Modèle,Famille,Version_données,Fichier,Temps_entraînement_secondes,Nombre_variables
0,Logistic Regression,Logistic Regression,Original,logistic_regression.joblib,1.2450,556
1,Decision Tree - Entropy,Decision Tree - Entropy,Original,decision_tree_entropy.joblib,0.1310,556
2,Decision Tree - Gini,Decision Tree - Gini,Original,decision_tree_gini.joblib,0.1133,556
3,Random Forest,Random Forest,Original,random_forest.joblib,2.7591,556
4,Logistic Regression - PCA,Logistic Regression,PCA,logistic_regression_pca.joblib,0.0678,9
5,Decision Tree - Entropy - PCA,Decision Tree - Entropy,PCA,decision_tree_entropy_pca.joblib,0.3581,9
6,Decision Tree - Gini - PCA,Decision Tree - Gini,PCA,decision_tree_gini_pca.joblib,0.3143,9
7,Random Forest - PCA,Random Forest,PCA,random_forest_pca.joblib,17.1838,9


**Ce que fait cette cellule :**
1. Pour chaque modèle défini plus haut, choisit les bonnes données d'entraînement (Original ou PCA)
2. Chronomètre l'entraînement avec `time.perf_counter()`
3. Sauvegarde le modèle entraîné en `.joblib` avec un nom de fichier généré automatiquement
4. Construit un **manifeste** (`model_manifest.csv`) qui résume tous les modèles entraînés :
   leur nom, leur famille d'algorithme, la version de données utilisée, le fichier associé,
   le temps d'entraînement et le nombre de variables utilisées

## Conclusion

Les 8 modèles (4 algorithmes × 2 versions de données) ont été entraînés avec le même découpage
Train/Test et des hyperparamètres comparables. Le manifeste `Modeles/model_manifest.csv` conserve
la famille du modèle, la version des données, le nombre de prédicteurs et le temps d'entraînement.

Le notebook suivant déterminera si la forte réduction du nombre de prédicteurs obtenue par PCA
compense une éventuelle perte de performance ou d'interprétabilité.

➡️ Suite dans **`NB_04_Evaluation.ipynb`**.
